# ORCA Analyst — QLoRA SFT (RAG format) trên Colab T4

**Mục tiêu:** fine-tune `Qwen/Qwen2.5-7B-Instruct` theo format ORCA
`(NARRATIVE + CONTEXT JSON + TÀI LIỆU TRUY XUẤT) → research voice`

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**

Sau train: lưu adapter → vLLM → ORCA `AI_BASE_URL` + `AI_MODEL_ANALYSIS=orca-analyst-v1`

## 0. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Cài thư viện

In [ ]:
!pip install -q -U transformers==4.51.3 peft==0.15.2 trl==0.15.2 bitsandbytes==0.45.4 accelerate datasets sentencepiece protobuf

## 2. Tải dataset seed từ GitHub
https://raw.githubusercontent.com/duyanhphan13579dz-dot/Orca-Multi-Finance/main/datasets/llm-human-voice/samples/sft_rag_train.jsonl

In [ ]:
import urllib.request
from pathlib import Path
URL = 'https://raw.githubusercontent.com/duyanhphan13579dz-dot/Orca-Multi-Finance/main/datasets/llm-human-voice/samples/sft_rag_train.jsonl'
DATA_PATH = Path('sft_rag_train.jsonl')
urllib.request.urlretrieve(URL, DATA_PATH)
print('lines:', sum(1 for _ in open(DATA_PATH, encoding='utf-8')))

## 3. JSONL → chat messages

In [ ]:
import json
from pathlib import Path
SYSTEM = '''Bạn là ORCA Agent — trợ lý phân tích tài chính.
Chỉ dùng số liệu trong CONTEXT/NARRATIVE và TÀI LIỆU TRUY XUẤT.
Không bịa số, không khuyến nghị mua/bán tuyệt đối.
Giọng senior equity research analyst Việt Nam.'''

def format_retrieved(retrieved):
    if not retrieved:
        return '(Không có đoạn tài liệu bổ sung.)'
    parts = []
    for i, p in enumerate(retrieved, 1):
        parts.append(f"[{i}] source={p.get('source','?')} {p.get('title','')}\n{p.get('content','')}")
    return '\n\n'.join(parts)

def to_messages(rec):
    q = rec['question']
    narrative = rec.get('context_narrative') or ''
    contract = rec.get('context_contract') or {}
    retrieved = rec.get('retrieved') or []
    answer = rec.get('preferred_answer') or rec.get('preferred') or ''
    user = (
        f'CÂU HỎI: {q}\n\n'
        f'NARRATIVE HỆ THỐNG:\n{narrative[:10000]}\n\n'
        f'CONTEXT JSON (rút gọn):\n{json.dumps(contract, ensure_ascii=False)[:12000]}\n\n'
        f'TÀI LIỆU TRUY XUẤT:\n{format_retrieved(retrieved)}\n\n'
        'Hãy tổng hợp phân tích chuyên sâu, bám số liệu quant và tài liệu trên.'
    )
    return {'messages': [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user},
        {'role': 'assistant', 'content': answer},
    ]}

rows = []
for line in Path('sft_rag_train.jsonl').read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if line:
        rows.append(to_messages(json.loads(line)))
Path('sft_rag_chat.jsonl').write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + '\n', encoding='utf-8')
print(f'chat examples: {len(rows)}')

## 4. Load Qwen2.5-7B 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
BASE_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model.config.use_cache = False
print('loaded', BASE_MODEL)

## 5. LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Dataset

In [ ]:
from datasets import load_dataset
raw = load_dataset('json', data_files='sft_rag_chat.jsonl', split='train')
print(raw)

## 7. Train SFTTrainer
Seed ~5 mẫu = smoke test. Production: 150–400 mẫu, epochs 2–3.

In [ ]:
from trl import SFTTrainer, SFTConfig
def formatting_func(example):
    return tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)
training_args = SFTConfig(
    output_dir='./orca-analyst-lora-out',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=1,
    save_strategy='epoch',
    bf16=True,
    optim='paged_adamw_8bit',
    max_seq_length=2048,
    packing=False,
    report_to='none',
)
trainer = SFTTrainer(model=model, args=training_args, train_dataset=raw, processing_class=tokenizer, formatting_func=formatting_func)
trainer.train()
print('train done')

## 8. Lưu adapter

In [ ]:
OUT = 'orca-analyst-lora'
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
!ls -lh orca-analyst-lora | head

## 9. Smoke test generate

In [ ]:
model.eval()
test_messages = raw[0]['messages'][:2]
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, temperature=0.4, top_p=0.9, do_sample=True)
text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(text[:1500])

## 10. Download adapter

In [ ]:
!zip -r orca-analyst-lora.zip orca-analyst-lora
from google.colab import files
files.download('orca-analyst-lora.zip')

## 11. Host vLLM (RunPod / VPS GPU)
```bash
vllm serve Qwen/Qwen2.5-7B-Instruct --enable-lora --lora-modules orca-analyst-v1=./orca-analyst-lora --host 0.0.0.0 --port 8000
```

## 12. ORCA `.env`
```bash
AI_BASE_URL=https://YOUR_PUBLIC_HOST/v1
AI_API_KEY=sk-local
AI_MODEL_ANALYSIS=orca-analyst-v1
AI_MODEL_ANALYSIS_FALLBACKS=qwen/qwen3.8-27b:free,inclusionai/ling-3.0-flash-fin:free
OPENROUTER_API_KEY=sk-or-v1-...
```

Seed 5 mẫu chỉ smoke-test. Annotate 150–400 mẫu trước train production. Xem `docs/RAG_PHASE_B.md`.